# 09 Delhi air: station and model layers, and what is blocked

Question (T165 only): what Delhi air-quality data can the product show today, and can the CAMS
model estimate be compared with measured pollution? **No to both.** The station source, S24 (CPCB
via data.gov.in), is blocked: its resource page was unreadable, so the resource ID, licence,
fields, units and averaging interval are unread and there are 0 station rows. The model source,
S04 (CAMS via Open-Meteo), is available and has labelled units, but it is enabled for Karnataka
only and `dl.yaml` does not list it. M14 (PM2.5 forecast) is blocked and shows the word "blocked"
and its planned, not started, test window; it shows no number.

So this notebook is an **audit**. It records the S24 and S04 field maps, the CAMS basis (pollutant,
unit, averaging) from the S04 record, and the two real results: the comparison is unavailable (no
measured basis) and the route exposure is unavailable (no route, no station value). Station
observations and the gridded model are separate layers and are never merged. Route exposure, when
it can be computed, is a time-weighted ambient concentration, never an inhaled dose. It draws no
chart because there is nothing to plot. The rules are tested on hand-built inputs in
`tests/test_delhi_product.py`, not shown as Delhi data.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/09_delhi_air/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from atlas.analysis.prices import audit_rows, load_record, source_status
from atlas.analysis.route_exposure import (
    Basis,
    blocked_card,
    blocked_view,
    compare_layers,
    route_exposure,
)
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.ingest.base import SourceUnavailable
from atlas.ingest.s04 import UNIT
from atlas.ingest.s24 import S24

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/09_delhi_air"))
OUT.mkdir(parents=True, exist_ok=True)
dl = load_state(Path("configs/states/dl.yaml"))
records = {s: load_record(Path(f"configs/sources/{s.lower()}.yaml")) for s in ("S24", "S04")}
m14_config = load_record(Path("configs/models/m14.yaml"))
m14_status = json.loads(Path("models/registry/m14.json").read_text("utf-8"))


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

In [ ]:
fields = [r for rec in records.values() for r in audit_rows(rec)]
status = [source_status(rec) for rec in records.values()]
by_status = {s: Counter(r["unit_status"] for r in fields if r["source"] == s) for s in records}
write_csv("fields.csv", fields)
write_csv("source_status.csv", status)
for s in status:
    print(s["source"], s["outcome"], "licence", s["licence"], "units", s["units"])
print({s: dict(c) for s, c in by_status.items()})

In [ ]:
# The S24 adapter refuses before any I/O; a frozen evidence record is only the argument it gets.
manifest = json.loads(Path("tests/fixtures/s24/manifest.json").read_text("utf-8"))
e = manifest["evidence"][0]
raw = RawArtifact(
    sha256=e["sha256"],
    path=Path("data/raw") / e["raw_path"],
    fetched_at=e["fetched_at"],
    media_type=e["media_type"],
    response_status=e["status"],
    source_url=e["url"],
    schema_mapping_version="s24-v1",
)
try:
    rows = S24().parse(raw, dl.source("S24"))
except SourceUnavailable as exc:
    adapter_error = str(exc)
else:
    raise AssertionError(f"S24 returned {rows.num_rows} rows; it must be blocked")
s04_in_dl = any(s.source_id == "S04" for s in dl.sources)
print(adapter_error, "| S04 listed in dl.yaml:", s04_in_dl)

In [ ]:
# CAMS basis from the S04 record: pm2_5 in the record's unit, an instant value (0 min average).
# There is no measured station basis (S24 unread), so nothing can be compared or weighted.
cams = Basis("model", "pm2_5", UNIT, 0)
assert UNIT in records["S04"]["units"]["detail"]
comparison = compare_layers(None, cams)
exposure = route_exposure(None)
print(comparison["status"], "|", comparison["reason"])
print(exposure["status"], "|", exposure["reason"])

In [ ]:
# The view the AirQuality component reads: two separate unavailable layers, no comparison, and a
# blocked M14 card (planned window, no score).
view = {
    "station": {"layer": "station"} | blocked_view(records["S24"]),
    "model": {
        "layer": "model",
        "state": "unavailable",
        "source_id": "S04",
        "reason": "S04 is available but not configured for Delhi: dl.yaml lists no S04 entry",
        "retry_condition": "Add S04 to dl.yaml with Delhi points and an adapter test.",
    },
    "comparison": comparison,
    "card": blocked_card(m14_status, m14_config),
}
(OUT / "air_view.json").write_text(json.dumps(view, indent=2) + "\n", "utf-8", newline="\n")
print(view["station"]["state"], view["model"]["state"], view["card"]["status"])

In [ ]:
summary = {
    "sources": {
        s["source"]: s
        | {
            "fields": sum(by_status[s["source"]].values()),
            "fields_by_unit_status": dict(sorted(by_status[s["source"]].items())),
        }
        for s in status
    },
    "s24_enabled_in_dl": dl.source("S24").enabled,
    "s04_listed_in_dl": s04_in_dl,
    "adapter_error": adapter_error,
    "station_rows": 0,
    "cams_basis": asdict(cams),
    "comparison": comparison,
    "route_exposure": exposure,
    "view": view,
    "charts": [],
    "limits": [
        "S24 is blocked: no station value, unit or averaging interval was read; no chart is drawn.",
        "S04 is a gridded model estimate (provenance model), never a station observation, and"
        " dl.yaml does not list it, so it is not shown for Delhi.",
        "The CAMS value is instantaneous; a time-weighted route concentration needs a value that"
        " represents a time span, and nothing is interpolated.",
        "Route exposure is an ambient concentration, never an inhaled dose.",
        "Coverage windows come from configs/models/m14.yaml; they are planned, not started, and"
        " shift as a whole if approval is late.",
        "The comparison and exposure rules are tested on hand-built inputs only.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))